#⚙️ **Automating Data Extraction & Locking the Raw Dataset**

###Objective:
This phase is all about leveling up efficiency by automating the entire data collection workflow. Rather than manually gathering military statistics for 140+ countries, a Python automation script is deployed to seamlessly navigate through the Global Firepower Index.

The script dynamically processes a list of country-specific URLs, pulls critical metrics — including manpower strength, defense expenditure, and equipment inventories — and consolidates everything into a structured raw dataset. The result is a faster, smarter, and more scalable data extraction pipeline that lays a solid foundation for deeper analysis.



In [ ]:
import pandas as pd
import requests
import re
import time
from bs4 import BeautifulSoup

In [ ]:
# 1. Get Country List
countries_url = "https://www.globalfirepower.com/countries-listing.php"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(countries_url, headers=headers, timeout=10)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

country_tags = soup.find_all("a", href=True)
ids = []

for a in country_tags:
    href = str(a["href"])
    if "country-military-strength-detail.php?country_id=" in href:
        cid = href.split("country_id=")[1]
        ids.append(cid)

ids = list(set(ids))

# 2. Key Metric Mapping
key_map = {
    "total_population": "Total Population:",
    "gdp": "Purchasing Power Parity:",
    "defense_budget": "Defense Budget:",
    "total_manpower": "Available Manpower",
    "active_personnel": "Active Personnel",
    "reserve_personnel": "Reserve Personnel",
    "air_force_personnel": "Air Force Personnel*",
    "army_personnel": "Army Personnel*",
    "navy_personnel": "Navy Personnel*",
    "total_aircraft": "Aircraft Total:",
    "attack_aircraft": "Attack Types:",
    "fighter_aircraft": "Fighters:",
    "transport_aircraft": "Transports (Fixed-Wing):",
    "helicopters": "Helicopters:",
    "special_mission_aircraft": "Special-Mission:",
    "trainer_aircraft": "Trainers:",
    "attack_helicopters": "Attack Helicopters:",
    "tanker_aircraft": "Tanker Fleet:",
    "naval_assets": "Total Assets:",
    "aircraft_carriers": "Aircraft Carriers:",
    "helicopter_carriers": "Helicopter Carriers:",
    "destroyers": "Destroyers:",
    "frigates": "Frigates:",
    "corvettes": "Corvettes:",
    "submarines": "Submarines:",
    "patrol_vessel": "Patrol Vessels:",
    "mine_warfare": "Mine Warfare:",
    "tanks": "Tanks:",
    "self_propelled_artillery": "Self-Propelled Artillery:",
    "towed_artillery": "Towed Artillery:",
    "rocket_artillery": "MLRS (Rocket Artillery):",
    "external_debt": "External Debt:"
}

# 3. Automation Function
def automation(ids, key_map):
    base_url = "https://www.globalfirepower.com/country-military-strength-detail.php?country_id="
    all_countries = {}

    for cid in ids:
        try:
            url = base_url + cid
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")

            country_name = cid.replace("-", " ").title()

            # Extract Power Index
            power_index = None
            info_block = soup.select_one("span.textNormal.textDkGray")
            if info_block:
                match = re.search(r"\d+\.\d+", info_block.text)
                if match:
                    power_index = match.group()

            # Extract Specifications
            specs_containers = soup.find_all("div", class_="specsGenContainers")
            data_found = {}

            for container in specs_containers:
                tag = container.select_one("span.textLarge.textBold.textShadow")
                value_tags = container.select("span.textWhite.textShadow")

                if tag and value_tags:
                    name = tag.text.strip()
                    values = [v.text.strip() for v in value_tags]
                    data_found[name] = values[-1]

            # Map the results
            result = {}
            for new_key, old_key in key_map.items():
                if old_key in data_found:
                    result[new_key] = data_found[old_key]
                else:
                    result[new_key] = None

            result["power_index"] = power_index
            all_countries[country_name] = result

            print(f"Scraped: {country_name}")
            time.sleep(1)

        except Exception as e:
            print(f"Error scraping {cid}: {e}")

    return all_countries

# 4. Execution
all_countries_data = automation(ids, key_map)

df = pd.DataFrame.from_dict(all_countries_data, orient="index")
df.reset_index(inplace=True)
df.rename(columns={"index": "country"}, inplace=True)

df.to_csv("military_raw_data.csv", index=False)
print("Done. File saved as military_raw_data.csv")

**Takeaway:** Efficiently scraped raw text-based military metrics for 145 countries and consolidated the unstructured information into a single export file — `military_raw_data.csv` — creating a centralized starting point for downstream processing and analysis.